## HOW TO LOAD THE DATA SO THAT OUR MODEL IS WELL TRAINED AND GENERALIZED - OPTIMIZATION TECHNIQUES

The three most widely used gradient descent methods are **Batch Gradient Descent**, **Stochastic Gradient Descent (SGD)**, and **Mini-batch Gradient Descent**, with the latter being the most popular choice in modern deep learning due to its balance of speed and stability.

*   **Batch Gradient Descent**: Uses the **entire dataset** to calculate the gradient and update parameters once per epoch; it is stable and precise but computationally expensive for large datasets.
*   **Stochastic Gradient Descent (SGD)**: Updates parameters for **each individual data point**, making it very fast and memory-efficient, though it produces noisier, more fluctuating convergence.
*   **Mini-batch Gradient Descent**: Divides data into **small subsets (batches)** (e.g., 32–256 samples) to compute gradients; it reduces the variance of SGD while leveraging optimized matrix operations, offering the best trade-off for most neural network training.

Beyond these fundamental variants, **adaptive optimizers** like **Adam** are frequently used in practice. Adam combines momentum and adaptive learning rates, allowing for faster convergence and better handling of sparse gradients, making it a top choice for tasks like NLP and complex deep learning models.



## When to Use Each Method and Their Trade-offs

### 1. Batch Gradient Descent
**When to Use:** Ideal for **small datasets** (typically <2,000 samples) that fit entirely in memory, or when a **precise, stable convergence** path is required for convex problems. It is rarely used in modern deep learning due to computational constraints.

*   **Advantages:**
    *   **Stability:** Provides a smooth, noiseless convergence path because it uses the true gradient of the entire dataset.
    *   **Vectorization:** Fully leverages matrix operations for efficient computation on small data.
    *   **Guaranteed Convergence:** For convex error surfaces, it is guaranteed to find the global minimum.
*   **Disadvantages:**
    *   **Computational Cost:** Extremely slow for large datasets as it processes all data for a single update.
    *   **Memory Intensive:** Requires the entire dataset to be loaded into RAM.
    *   **Local Minima:** Lacks the noise necessary to escape shallow local minima in non-convex landscapes.

### 2. Stochastic Gradient Descent (SGD)
**When to Use:** Best for **online learning** (streaming data), **massive datasets** that cannot fit in memory, or when the model needs to **escape local minima** due to a noisy loss landscape.

*   **Advantages:**
    *   **Speed:** Updates parameters immediately after every single sample, allowing for rapid initial progress.
    *   **Memory Efficiency:** Requires minimal memory as only one sample is processed at a time.
    *   **Escape Local Minima:** The inherent noise in updates can help jump out of poor local minima, potentially finding better solutions in non-convex problems.
*   **Disadvantages:**
    *   **High Variance:** The convergence path is noisy and fluctuates wildly, making it difficult to settle exactly at the minimum.
    *   **Loss of Vectorization:** Processing one sample at a time prevents efficient use of GPU parallelization.
    *   **Instability:** Requires careful learning rate scheduling (annealing) to converge effectively.

### 3. Mini-batch Gradient Descent
**When to Use:** The **standard default** for almost all deep learning tasks. It is optimal when you have **GPU acceleration** and a dataset large enough to require batching but small enough to fit subsets in memory (typical batch sizes: 32, 64, 128).

*   **Advantages:**
    *   **Balance:** Combines the stability of Batch GD with the speed of SGD.
    *   **Hardware Efficiency:** Allows for optimized matrix operations on GPUs, significantly speeding up training compared to pure SGD.
    *   **Reduced Variance:** Averaging gradients over a batch reduces noise, leading to smoother convergence than SGD.
*   **Disadvantages:**
    *   **Hyperparameter Tuning:** Introduces the "batch size" as a new hyperparameter that needs tuning.
    *   **Complexity:** Slightly more complex to implement than pure batch or stochastic methods.

### 4. Adaptive Optimizers (e.g., Adam, AdamW)
**When to Use:** Recommended for **complex architectures** (Transformers, CNNs), **sparse data** (NLP, recommendation systems), or when you need **fast convergence** with minimal hyperparameter tuning. Adam is often the default choice in frameworks like PyTorch and TensorFlow.

*   **Advantages:**
    *   **Adaptive Learning Rates:** Automatically adjusts learning rates for each parameter, handling sparse gradients and varying scales effectively.
    *   **Momentum:** Incorporates momentum to accelerate convergence and dampen oscillations.
    *   **Robustness:** Less sensitive to the initial learning rate choice compared to standard SGD.
*   **Disadvantages:**
    *   **Generalization:** In some specific computer vision tasks, well-tuned SGD with momentum may generalize slightly better than Adam.
    *   **Memory Overhead:** Requires storing additional moments (first and second) for each parameter, increasing memory usage.

## Summary Comparison

| Method | Best Use Case | Primary Advantage | Primary Disadvantage |
| :--- | :--- | :--- | :--- |
| **Batch GD** | Small, convex datasets | Stable, exact gradient | Too slow for large data |
| **SGD** | Online learning, huge data | Fast updates, escapes local minima | Noisy, unstable convergence |
| **Mini-batch** | **Standard Deep Learning** | **Best speed/stability trade-off** | Requires batch size tuning |
| **Adam** | Complex models, sparse data | Fast convergence, low tuning | Higher memory usage |



---

## Dataset Class

The Dataset class is essentially a blueprint. When you create a
custom Dataset, you decide how data is loaded and returned.

It defines:
* **__init__()**: which tells how data should be loaded.
* **__len__()**: which returns the total number of samples.
* **__getitem__(index)**: which returns the data (and label) at the
given index.

---

## DataLoader Class

The DataLoader wraps a Dataset and handles batching, shuffling,
and parallel loading for you.

DataLoader Control Flow:

* At the start of each epoch, the DataLoader (if *shuffle=True*)
shuffles indices(using a sampler).
* It divides the indices into chunks of batch_size.
* for each index in the *chunk*, data samples are fetched from
the *Dataset object*
* The samples are then collected and combined into a batch
(using **collate_fn**)
* The batch is returned to the main training loop.

## Manually applying 
**Problem in manual approch**
1. No standard interface for data
2. No easy way to apply transformations
3. Shuffling and sampling
4. Batch management & Parallelization

In [1]:
batch_size = 32
epochs = 25
n_samples = 200 #len(X_train_tensor) or 200

In [ ]:
for epoch in range(epochs):
    # looping over the dataset in chunks of 'batch_size'
    for start_idx in range(0, n_samples, batch_size):
        print(start_idx)
        end_idx = start_idx + batch_size
        print(end_idx, '\n')
        X_batch = X_train_tensor[start_idx:end_idx]
        Y_batch = Y_train_tensor[start_idx:end_idx]
        
        # forward pass
        y_pred = model(X_batch)
        loss = loss_function(y_pred, y_batch.view(-1, 1))
        
        # update step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch: {epoch+1} | Loss: {loss.item()}')
        

## Using Torch Dataset Class and Dataloader Class to apply mini batch

In [ ]:
import torch
from sklearn.datasets import make_classification

In [2]:
x, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=342    # random state for reproducibility
)

In [3]:
print(f'Training featues:\n{x}\n\n')
print(f'Target feature\n{y}')

Training featues:
[[-1.35965861 -0.00519508]
 [ 1.12403749 -1.52562649]
 [ 2.42844288 -0.32932538]
 [-0.04772659  1.56863564]
 [-0.74203069 -0.55899951]
 [ 0.79121111 -1.29494682]
 [-1.16662006 -1.46391552]
 [-1.06077661  1.63304789]
 [ 1.14522708 -0.63336163]
 [-2.22996483  1.3803021 ]]


Target feature
[1 0 1 1 0 0 0 1 0 1]


In [4]:
x.shape, y.shape

((10, 2), (10,))

In [5]:
# convert data to PyTorch tensors
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [6]:
# import Dataset and DataLoader classes
from torch.utils.data import Dataset, DataLoader

## PyTorch DataLoader & Its Parameters

The DataLoader class in PyTorch comes with several parameters that allow you to customize how data is loaded, batched, and preprocessed. Some of the most commonly used and important parameters include:

#### 1. dataset (mandatory)

- The Dataset from which the DataLoader will pull data.
- Must be a subclass of `torch.utils.data.Dataset` that implements `__getitem__` and `__len__`.

#### 2. batch_size

- How many samples per batch to load.
- Default is 1.
- Larger batch sizes can speed up training on GPUs but require more memory.

#### 3. shuffle

- If `True`, the DataLoader will shuffle the dataset indices each epoch.
- Helpful to avoid the model becoming too dependent on the order of samples.

#### 4. num_workers

- The number of worker processes used to load data in parallel.
- Setting `num_workers > 0` can speed up data loading by leveraging multiple CPU cores, especially if I/O or preprocessing is a bottleneck.

#### 5. pin_memory

- If `True`, the DataLoader will copy tensors into pinned (page-locked) memory before returning them.
- This can improve GPU transfer speed and thus overall training throughput, particularly on CUDA systems.

#### 6. drop_last

- If `True`, the DataLoader will drop the last incomplete batch if the total number of samples is not divisible by the batch size.
- Useful when exact batch sizes are required (for example, in some batch normalization scenarios).

#### 7. collate_fn

- A callable that processes a list of samples into a batch (the default simply stacks tensors).
- Custom `collate_fn` can handle variable-length sequences, perform custom batching logic, or handle complex data structures.

#### 8. sampler

- `sampler` defines the strategy for drawing samples (e.g., for handling imbalanced classes, or custom sampling strategies).
- `batch_sampler` works at the batch level, controlling how batches are formed.
- Typically, you don't need to specify these if you are using `batch_size` and `shuffle`. However, they provide lower-level control if you have advanced requirements.

In [7]:
class CustomData(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    # returns the lenght of the data
    def __len__(self):
        return self.features.shape[0]

    # retures the data row of given index
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [8]:
dataset = CustomData(x,y)

In [9]:
len(dataset)

10

In [12]:
print(dataset[0])
print(dataset[4])

(tensor([-1.3597, -0.0052]), tensor(1.))
(tensor([-0.7420, -0.5590]), tensor(0.))


DataLoader is iterable

In [32]:
dataloader = DataLoader(dataset=dataset, batch_size=2)
dataloader2 = DataLoader(dataset=dataset, batch_size=2, shuffle=True)

In [33]:
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print('-'*30)

tensor([[-1.3597, -0.0052],
        [ 1.1240, -1.5256]])
tensor([1., 0.])
------------------------------
tensor([[ 2.4284, -0.3293],
        [-0.0477,  1.5686]])
tensor([1., 1.])
------------------------------
tensor([[-0.7420, -0.5590],
        [ 0.7912, -1.2949]])
tensor([0., 0.])
------------------------------
tensor([[-1.1666, -1.4639],
        [-1.0608,  1.6330]])
tensor([0., 1.])
------------------------------
tensor([[ 1.1452, -0.6334],
        [-2.2300,  1.3803]])
tensor([0., 1.])
------------------------------


In [34]:
for batch_features, batch_labels in dataloader2:
    print(batch_features)
    print(batch_labels)
    print('-'*30)

tensor([[ 2.4284, -0.3293],
        [-2.2300,  1.3803]])
tensor([1., 1.])
------------------------------
tensor([[-1.1666, -1.4639],
        [ 0.7912, -1.2949]])
tensor([0., 0.])
------------------------------
tensor([[-1.0608,  1.6330],
        [-0.0477,  1.5686]])
tensor([1., 1.])
------------------------------
tensor([[ 1.1240, -1.5256],
        [-1.3597, -0.0052]])
tensor([0., 1.])
------------------------------
tensor([[ 1.1452, -0.6334],
        [-0.7420, -0.5590]])
tensor([0., 0.])
------------------------------


#### We can add parallelization using the **workers** in the DataLoader

So the problems are solved
1. Standard interface is solved by Dataset class
2. Transformation can be done in Dataset Class
3. Shuffling and sampling can be by DataLoader Class
4. Batch Management & Parallalization is done by DataLoader Class

---
### **Samplers** in *DataLoader* class

In PyTorch, the sampler in the DataLoader determines the strategy for selecting samples from
the dataset during data loading. It controls how indices of the dataset are drawn for each
batch.

#### Types of Samplers

PyTorch provides several predefined samplers, and you can create custom ones:

1. SequentialSampler:
    * Samples elements sequentially, in the order they appear in the dataset.
    * Default when shuffle=False.

2. RandomSampler:
    * Samples elements randomly without replacement.
    * Default when shuffle=True.

3. Our own custom Sampler:
    * we create our custom sampler according to the use case or scenrio
    * *Case*: When there is *imbalanced* data e.g. 99% True data and 1% False data then when we create batch we must keep the 99:1 ratio in each batch for fair performace if in most batch there are 100% True data then this will significantly effect the model performance.

---

### **collate_fn**:
The collate_fn in PyTorch's DataLoader is a function that specifies **how to combine a list of samples from a dataset into a single batch**. By default, the DataLoader uses a simple batch collation mechanism, but collate_fn allows you to customize how the data should be processed and batched.

**Usecase**: let say we tokenzined words to train on them but the token may have diffenent sizes so if we try to collate them it will trough size-missmatch error. Thus there we will need custom collate_fn so that we can make it work by fixing the size issue by either transforming the data or other methods.